# TrustHire — train / fine-tune voice anti-spoofing on Kaggle/Colab

ASVspoof 2019 LA is openly downloadable (Edinburgh DataShare, ODC-By). Compare fine-tuned SSL front-ends (WavLM / XLS-R) with AASIST and RawNet2; pick the best by **In-the-Wild EER** (cross-domain).

In [ ]:
# --- environment (Kaggle: enable GPU T4x2 / P100; Colab: Runtime > GPU) ---
!pip -q install uv
!git clone https://github.com/<you>/trusthire.git trusthire || true
%cd trusthire
!uv pip install --system -e . 2>&1 | tail -2
!python -m trusthire.cli download-models --profile gpu

In [ ]:
!wget -q https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip && unzip -q LA.zip -d /kaggle/working/

In [ ]:
for m in ['rawnet2', 'aasist']:
    !python -m trusthire.audio.train model={m} data.asvspoof_root=/kaggle/working/LA epochs=20 batch_size=32
!python -m trusthire.audio.train model=ssl encoder=microsoft/wavlm-base-plus data.asvspoof_root=/kaggle/working/LA epochs=6 batch_size=16
!python -m trusthire.audio.train model=ssl encoder=facebook/wav2vec2-xls-r-300m data.asvspoof_root=/kaggle/working/LA epochs=4 batch_size=8

## Evaluate against the shipped models (In-the-Wild + ASVspoof19 LA eval, with degradations)

In [ ]:
!python -m trusthire.audio.eval --datasets itw asv19la --n 1000 --backends spectra0 aasist --conditions clean opus_16k noise_snr10 telephone